# CPMK 3 — Recovery V2 pelatihan Hard-3
Notebook ini menilai tiga kelas target, yaitu `bed`, `book`, dan `cabinet-merged`. Checkpoint dan hasil evaluasi validation YOLO11n serta YOLOv8n dipulihkan dari run Hard-3 pertama. Mask R-CNN dilatih kembali dari bobot pralatih COCO.
Pada run awal (AMP, laju belajar 0,005), loss menjadi nonfinite pada epoch 1 batch 32. Recovery FP32 dengan laju belajar yang sama gagal pada batch 43. Gambar train ID 153 dan 205 memiliki target Hard-3 kosong. `loss_classifier` dan `loss_box_reg` bernilai NaN, sementara `loss_objectness` mencapai 933,693. Jumlah proposal, gradien, dan kondisi parameter saat itu tidak tercatat.
Versi ini mempertahankan FP32, menurunkan laju belajar menjadi 0,0005, dan membatasi norma L2 gradien pada 5,0. Protokol COCOeval tetap mengikuti evaluasi validation sebelumnya. FP32, laju belajar, dan clipping merupakan perubahan terhadap baseline, sehingga efek perubahan cakupan kelas pada Mask R-CNN perlu ditafsirkan bersama perbedaan konfigurasi tersebut.


## 1. Dependensi dan lingkungan

PyTorch dan Torchvision memakai pasangan yang disediakan Kaggle. Ultralytics dipatok ke versi baseline. Versi aktual dan GPU dicatat agar perbedaan lingkungan dapat ditelusuri.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ultralytics==8.4.216", "pycocotools==2.0.10", "pyyaml"],
    check=True,
)

import csv
import gc
import hashlib
import io
import json
import math
import random
import re
import shutil
import time
import traceback
from collections import Counter, defaultdict
from contextlib import contextmanager, redirect_stdout
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from zipfile import BadZipFile, ZIP_DEFLATED, ZIP_STORED, ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import ultralytics
import yaml
from IPython.display import display
from PIL import Image
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights, maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms.functional import to_tensor
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan satu GPU Tesla T4 di pengaturan Kaggle.")
DEVICE = torch.device("cuda:0")
assert ultralytics.__version__ == "8.4.216", "Mulai ulang kernel setelah mengganti versi Ultralytics."
torchvision.ops.nms(torch.tensor([[0., 0., 3., 3.]], device=DEVICE),
                    torch.ones(1, device=DEVICE), 0.5)

## 2. Konfigurasi recovery
Lampirkan Panoptic Indoor v5 COCO, hasil gabungan Hard-3 pertama, serta artefak unified COCOeval all-class. Sumber recovery bisa berupa ZIP asli atau folder yang diekstrak Kaggle. Notebook membaca sumber secara langsung. Jika beberapa input cocok, isi `RECOVERY_SOURCE_PATH` dengan path ZIP atau direktori run yang memuat `run_summary.json`.


In [ ]:
INPUT_ROOT = Path("/kaggle/input")
DATASET_ROOT = None                 # Isi jika ada beberapa salinan dataset.
BASELINE_SOURCE = None              # Folder dua berkas evaluasi atau ZIP evaluasi.
RUN_TRAIN = {"yolo11n": False, "yolov8n": False, "maskrcnn": True}
RUN_EVALUATION = True
RECOVERY_SOURCE_PATH = None         # ZIP atau direktori run hasil ekstraksi.

SEED = 42
CLASSES = {2: (1, "bed"), 4: (2, "book"), 8: (3, "cabinet-merged")}
NAMES = [name for _, name in CLASSES.values()]
EXPECTED = {"train": (397, 178, 60), "valid": (166, 40, 19), "test": (359, 530, 137)}
SOURCE_HASHES = {
    "train": "6426839e66216ad207e3f51f9521520256ec5aa33effe57a06a5d3c766b49117",
    "valid": "e0ca4ae5d12ce52a16445e9395a244424520c0b8c0c635201deb68257e474274",
    "test": "1b756f0cab2be19fb338e35aea80c3a1e40eca6c5343f4daee9bf3d675bd466f",
}
ROTATIONS = {
    ("train", "182169107_jpg.rf.c9b1bbecf1474a5b36a6df197c30dfed.jpg"): "CCW",
    ("train", "182169121_jpg.rf.eb0ccb71d0ea38adf1e38bf9c02a4a15.jpg"): "CCW",
    ("valid", "182169109_jpg.rf.0531e60eb6e404d9633ed0c23a381fe7.jpg"): "CW",
    ("valid", "182169108_jpg.rf.205d9f1ed8f888006b8f97fc29eb8790.jpg"): "CCW",
}
YOLO_WEIGHTS = {
    "yolo11n": ("yolo11n-seg.pt", "55ed65c56c91713d23e8402371c6c49a6fd84f257f7dce452e8d70e41dcbe152"),
    "yolov8n": ("yolov8n-seg.pt", "a7cd8f929e1903d78a12a48efecab430209f18dc46cb96c3599a5980c63c423c"),
}
YOLO_TRAIN = dict(
    epochs=60, patience=60, imgsz=640, batch=8, device=0, workers=2,
    amp=True, cache=False, seed=SEED, deterministic=True, val=True, plots=True,
    save=True, save_period=5, exist_ok=False, verbose=True, optimizer="AdamW",
    lr0=0.000128, momentum=0.9, weight_decay=0.0005, nbs=64, lrf=0.01,
    cos_lr=False, warmup_epochs=3.0, warmup_momentum=0.8, warmup_bias_lr=0.0,
    mosaic=1.0, close_mosaic=10,
)
RCNN_TRAIN = dict(epochs=20, batch=2, workers=2, lr=0.0005, momentum=0.9,
                  weight_decay=0.0005, milestones=[10, 15], gamma=0.1,
                  min_size=640, max_size=960, amp=False, save_period=5, max_minutes=105,
                  gradient_clip_max_norm=5.0)
PREFLIGHT_NEGATIVE_IDS = (153, 205)
RCNN_LOG_BATCHES = 100
PROTOCOL = dict(dataset_split="validation", num_images=166, confidence_threshold=0.001,
                max_detections=100, mask_threshold=0.5, yolo_image_size=640,
                yolo_nms_iou=0.7, maskrcnn_min_size=640, maskrcnn_max_size=960,
                maskrcnn_nms_iou=0.5)
TRAIN_BASELINE = {"yolo11n": (60, 599.949), "yolov8n": (60, 571.242),
                  "maskrcnn": (20, 2238.738107909)}
PIPELINE_BASELINE = {"yolo11n":31.81647763899997,"yolov8n":19.78087676599995,"maskrcnn":40.2276660120001}
ROOT = Path("/kaggle/working/cpmk3_hard3") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
ROOT.mkdir(parents=True, exist_ok=False)
DATA = ROOT / "dataset"
EXPERIMENTS = ROOT / "experiments"
EXPORTS = ROOT / "exports"
EXPORTS.mkdir()
MODEL_STATE = {key: {"training": "pending", "evaluation": "pending"} for key in RUN_TRAIN}
ACTIVE_RUNS = {}
MODEL_SPEC_HASHES = {}
RECOVERED_EVALUATIONS = {}
ENVIRONMENT = dict(torch=torch.__version__, torchvision=torchvision.__version__,
                   ultralytics=ultralytics.__version__, gpu=torch.cuda.get_device_name(0),
                   cuda=torch.version.cuda, seed=SEED)
EXPERIMENT_SPEC = dict(yolo=YOLO_TRAIN, maskrcnn=RCNN_TRAIN, classes=CLASSES,
                       source_hashes=SOURCE_HASHES, protocol=PROTOCOL)

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
                    encoding="utf-8")
    temp.replace(path)

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8-sig"))

def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def fingerprint(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(payload.encode()).hexdigest()

SPEC_HASH = fingerprint(EXPERIMENT_SPEC)
write_json(ROOT / "environment.json", ENVIRONMENT)
write_json(ROOT / "experiment_spec.json", EXPERIMENT_SPEC)

def save_status():
    write_json(ROOT / "status.json", MODEL_STATE)

def seed_all():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def release_gpu():
    gc.collect()
    torch.cuda.empty_cache()

save_status()
print(ROOT, ENVIRONMENT)


## 3. Identifikasi data

Anotasi ketiga split diperiksa terhadap versi baseline. Sel persiapan membaca original test untuk memeriksa struktur dan mempertahankan split. Semua loader dan evaluasi di bawah hanya memakai train dan validation.

In [ ]:
def discover_dataset(override, input_root):
    if override is not None:
        candidates = [Path(override)]
    else:
        candidates = sorted({p.parent.parent for p in input_root.rglob("_annotations.coco.json")
                             if p.parent.name == "train" and sha256(p) == SOURCE_HASHES["train"]})
    if len(candidates) != 1:
        raise RuntimeError(f"Perlu satu dataset COCO versi baseline. Atur DATASET_ROOT. Kandidat: {candidates}")
    root = candidates[0]
    for split, digest in SOURCE_HASHES.items():
        path = root / split / "_annotations.coco.json"
        if not path.is_file() or sha256(path) != digest:
            raise ValueError(f"Anotasi {split} berbeda dari baseline: {path}")
    return root

COCO_ROOT = discover_dataset(DATASET_ROOT, INPUT_ROOT)
print("Dataset:", COCO_ROOT)

## 4. Koreksi anotasi dan persiapan Hard-3

Empat gambar telah memiliki orientasi piksel yang benar. Karena ukuran pada metadata COCO tertukar, yang diputar adalah titik poligonnya. Poligon target yang sama menjadi sumber label YOLO dan COCO. ID COCO asli 2, 4, dan 8 dipetakan ke ID Hard-3 COCO 1, 2, dan 3. Untuk YOLO, ID tersebut dikurangi satu.

In [ ]:
def rotate_polygon(points, width, height, direction):
    x, y = points.T
    if direction == "CW":
        return np.column_stack((height - y, x))
    if direction == "CCW":
        return np.column_stack((y, width - x))
    if direction is not None:
        raise ValueError(f"Rotasi tidak dikenal: {direction}")
    return points.copy()

def normalize_polygon(xy, size):
    result = xy / np.asarray(size, dtype=float)
    if not np.isfinite(result).all() or (result < 0).any() or (result > 1).any():
        raise ValueError("Poligon di luar batas gambar.")
    return result

def prepare_dataset(source_root, out):
    categories = [{"id": new, "name": name, "supercategory": "indoor"}
                  for new, name in CLASSES.values()]
    splits, repairs = {}, []
    for split, expected in EXPECTED.items():
        src_dir = source_root / split
        images_dir = out / split / "images"
        labels_dir = out / split / "labels"
        images_dir.mkdir(parents=True, exist_ok=False)
        labels_dir.mkdir()
        original = read_json(src_dir / "_annotations.coco.json")
        source_names = {row["id"]: row["name"] for row in original["categories"]}
        assert all(source_names[old] == name for old, (_, name) in CLASSES.items())
        metadata, sizes, labels, corrected = {}, {}, defaultdict(list), []
        seen_stems, seen_ids = set(), set()
        for image in original["images"]:
            name = image["file_name"]
            assert Path(name).name == name and image["id"] not in metadata, image
            assert Path(name).stem not in seen_stems, name
            seen_stems.add(Path(name).stem)
            src = src_dir / name
            with Image.open(src) as pic:
                actual = pic.size
                # YOLO dan Torchvision harus melihat orientasi piksel yang sama.
                assert pic.getexif().get(274, 1) == 1, f"Orientasi EXIF tidak netral: {name}"
                pic.verify()
            nominal = image["width"], image["height"]
            direction = ROTATIONS.get((split, name))
            if direction:
                assert actual == nominal[::-1] and actual != nominal, name
                repairs.append(dict(split=split, image_id=image["id"], file_name=name,
                                    rotation=direction, coco_size=nominal, image_size=actual))
            else:
                assert actual == nominal, (name, actual, nominal)
            metadata[image["id"]] = dict(image, width=actual[0], height=actual[1])
            sizes[image["id"]] = nominal, actual, direction
            shutil.copy2(src, images_dir / name)
        for ann in original["annotations"]:
            old = ann["category_id"]
            if old not in CLASSES:
                continue
            assert ann["id"] not in seen_ids and ann.get("iscrowd", 0) == 0, ann["id"]
            seen_ids.add(ann["id"])
            image_id = ann["image_id"]
            nominal, actual, direction = sizes[image_id]
            new, _ = CLASSES[old]
            polygon = ann["segmentation"]
            # Baseline menerima satu polygon per instance. Tolak format berbeda.
            assert isinstance(polygon, list) and len(polygon) == 1, ann["id"]
            coords = polygon[0]
            assert len(coords) >= 6 and len(coords) % 2 == 0, ann["id"]
            xy = rotate_polygon(np.asarray(coords, dtype=np.float64).reshape(-1, 2),
                                *nominal, direction)
            normalized = normalize_polygon(xy, actual)
            segment = [xy.ravel().tolist()]
            width, height = actual
            rle = mask_utils.merge(mask_utils.frPyObjects(segment, height, width))
            area = float(mask_utils.area(rle))
            x, y, w, h = map(float, mask_utils.toBbox(rle))
            assert area > 0 and min(x, y) >= 0 and min(w, h) > 0
            assert x + w <= width and y + h <= height, ann["id"]
            corrected.append(dict(ann, category_id=new, segmentation=segment,
                                  bbox=[x, y, w, h], area=area, iscrowd=0))
            coordinates = " ".join(f"{x:.10f}" for x in normalized.ravel())
            labels[image_id].append(f"{new - 1} {coordinates}")
        for image_id, image in metadata.items():
            lines = labels.get(image_id, [])
            path = labels_dir / (Path(image["file_name"]).stem + ".txt")
            path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
        positives = len(labels)
        assert (len(metadata), len(corrected), positives) == expected, (split, expected)
        assert len(list(images_dir.iterdir())) == len(list(labels_dir.iterdir())) == expected[0]
        gt = dict(info=original.get("info", {}), licenses=original.get("licenses", []),
                  images=list(metadata.values()), categories=categories, annotations=corrected)
        write_json(out / split / "hard3.coco.json", gt)
        splits[split] = dict(images=len(metadata), instances=len(corrected),
                             positive_images=positives, negative_images=len(metadata)-positives,
                             instances_by_class=dict(Counter(a["category_id"] for a in corrected)))
    assert {(row["split"], row["file_name"]) for row in repairs} == set(ROTATIONS)
    yaml_data = dict(path=str(out), train="train/images", val="valid/images",
                     test="test/images", nc=3, names=NAMES)
    (out / "data.yaml").write_text(yaml.safe_dump(yaml_data, sort_keys=False), encoding="utf-8")
    manifest = dict(source=str(source_root), source_sha256=SOURCE_HASHES, splits=splits,
                    orientation_repairs=repairs, test_used_for_training_or_evaluation=False,
                    class_mapping=[dict(original_coco=old, original_yolo=old-1, hard3_coco=new,
                                        hard3_yolo=new-1, name=name)
                                   for old, (new, name) in CLASSES.items()])
    write_json(ROOT / "dataset_manifest.json", manifest)
    return manifest

MANIFEST = prepare_dataset(COCO_ROOT, DATA)
DATA_YAML = DATA / "data.yaml"
VAL_ANN = DATA / "valid/hard3.coco.json"

## 5. Pemeriksaan data dan contoh anotasi

Train memiliki 60 gambar positif dan 337 negatif. Validation memiliki 19 gambar positif dan 147 negatif. Gambar yang berisi objek di luar Hard-3 tetap digunakan. Untuk Mask R-CNN, gambar tanpa target memakai tensor kosong dengan dimensi yang sesuai.

In [ ]:
class IndoorCOCODataset(Dataset):
    def __init__(self, split):
        if split not in {"train", "valid"}:
            raise ValueError("Loader notebook dibatasi pada train dan validation.")
        self.image_dir = DATA / split / "images"
        self.coco = COCO(str(DATA / split / "hard3.coco.json"))
        self.ids = sorted(self.coco.getImgIds())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        image_id = self.ids[index]
        meta = self.coco.imgs[image_id]
        with Image.open(self.image_dir / meta["file_name"]) as picture:
            image = to_tensor(picture.convert("RGB"))
        anns = self.coco.imgToAnns[image_id]
        boxes, masks = [], []
        for a in anns:
            x, y, w, h = a["bbox"]
            boxes.append([x, y, x + w, y + h])
            masks.append(self.coco.annToMask(a))
        n = len(anns)
        target = dict(
            boxes=torch.tensor(boxes, dtype=torch.float32).reshape(n, 4),
            labels=torch.tensor([a["category_id"] for a in anns], dtype=torch.int64),
            masks=torch.from_numpy(np.stack(masks)).to(torch.uint8) if n else
                  torch.zeros((0, meta["height"], meta["width"]), dtype=torch.uint8),
            image_id=torch.tensor(image_id, dtype=torch.int64),
            area=torch.tensor([a["area"] for a in anns], dtype=torch.float32),
            iscrowd=torch.zeros(n, dtype=torch.int64),
        )
        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

VAL_DS = IndoorCOCODataset("valid")
example_ids = []
for cls in range(1, 4):
    example_ids.append(min(a["image_id"] for a in VAL_DS.coco.anns.values()
                           if a["category_id"] == cls))
example_ids.append(next(i for i in VAL_DS.ids if not VAL_DS.coco.imgToAnns[i]))
fig, axes = plt.subplots(2, 2, figsize=(9, 7), layout="constrained")
for axis, image_id in zip(axes.flat, example_ids):
    pixels, target = VAL_DS[VAL_DS.ids.index(image_id)]
    n = len(target["labels"])
    assert target["boxes"].shape == (n, 4)
    assert target["masks"].shape == (n, *pixels.shape[1:])
    axis.imshow(pixels.permute(1, 2, 0))
    for m in target["masks"]:
        axis.contour(m.numpy(), levels=[0.5], linewidths=1.3)
    names = ", ".join(NAMES[i-1] for i in target["labels"].tolist())
    axis.set_title(f"ID {image_id}: {names or 'tanpa instance target'}", fontsize=9)
    axis.axis("off")
fig.savefig(ROOT / "dataset_qa.png", dpi=150)
plt.show()
plt.close(fig)
display(pd.DataFrame.from_dict(MANIFEST["splits"], orient="index"))

## 6–7. Pemulihan YOLO11n dan YOLOv8n
Recovery membaca paket hasil awal, baik dalam ZIP maupun direktori yang telah diekstrak. Konfigurasi YOLO, bobot awal, pemetaan kelas, sumber anotasi, ground truth validation, dan checksum checkpoint dicocokkan. Spesifikasi historis disimpan bersama kedua YOLO agar perubahan konfigurasi Mask R-CNN dapat ditelusuri. Evaluasi historis digunakan kembali jika terkait dengan checkpoint dan protokol yang sama.


In [ ]:


def checkpoint_names(key):
    ext = "pth" if key == "maskrcnn" else "pt"
    return (f"weights/best.{ext}", f"weights/last.{ext}")

def validate_training_artifacts(key, run, expected_spec_hash=None):
    expected_spec_hash = SPEC_HASH if expected_spec_hash is None else expected_spec_hash
    needed = ["results.csv", "summary.json", "config.json", *checkpoint_names(key)]
    needed += ["training_curves.png"] if key == "maskrcnn" else ["results.png", "args.yaml"]
    for name in needed:
        path = run / name
        if not path.is_file() or path.stat().st_size == 0:
            raise FileNotFoundError(f"Artefak wajib: {path}")
    summary = read_json(run / "summary.json")
    expected = RCNN_TRAIN["epochs"] if key == "maskrcnn" else YOLO_TRAIN["epochs"]
    if (summary["model"] != key or summary["epochs_completed"] != expected
            or summary["spec_sha256"] != expected_spec_hash
            or not math.isfinite(summary["training_seconds"])
            or summary["training_seconds"] <= 0):
        raise ValueError(f"Run tidak lengkap atau spesifikasinya berbeda: {run}")
    if set(summary["checkpoint_sha256"]) != set(checkpoint_names(key)):
        raise ValueError(f"Daftar hash checkpoint tidak lengkap: {run}")
    for name, digest in summary["checkpoint_sha256"].items():
        if sha256(run / name) != digest:
            raise ValueError(f"Checksum checkpoint berbeda: {run / name}")

def archive_verified(destination, files):
    if destination.exists():
        raise FileExistsError(destination)
    temporary = destination.with_suffix(".zip.tmp")
    expected = {}
    with ZipFile(temporary, "x", allowZip64=True) as z:
        for name, path in files:
            assert name not in expected and path.is_file(), (name, path)
            expected[name] = path.stat().st_size
            z.write(path, arcname=name, compress_type=ZIP_STORED if path.suffix in
                    {".pt", ".pth", ".png", ".jpg"} else ZIP_DEFLATED)
    with ZipFile(temporary) as z:
        assert {entry.filename: entry.file_size for entry in z.infolist()} == expected
        bad = z.testzip()
        if bad is not None:
            raise IOError(f"CRC ZIP gagal: {bad}")
    temporary.rename(destination)

def run_entries(run, prefix):
    return [(f"{prefix}/{p.relative_to(run).as_posix()}", p)
            for p in sorted(run.rglob("*")) if p.is_file()]

def common_entries():
    return [(f"provenance/{n}", ROOT / n) for n in
            ("environment.json", "experiment_spec.json", "dataset_manifest.json", "dataset_qa.png")]



def next_run(key):
    parent = EXPERIMENTS / key
    parent.mkdir(parents=True, exist_ok=True)
    attempt = 1
    while (parent / f"attempt_{attempt:02d}").exists():
        attempt += 1
    return parent / f"attempt_{attempt:02d}"

def finish_summary(key, run, summary):
    summary.update(model=key, spec_sha256=SPEC_HASH,
                   checkpoint_sha256={n: sha256(run / n) for n in checkpoint_names(key)})
    write_json(run / "summary.json", summary)

@contextmanager
def recovery_reader(source):
    if source.is_dir():
        root = source.resolve()
        members = {p.relative_to(root).as_posix(): p for p in root.rglob("*") if p.is_file()}

        def open_member(name):
            path = members[name].resolve()
            if not path.is_relative_to(root):
                raise ValueError(f"Entry recovery keluar dari direktori sumber: {name}")
            return path.open("rb")

        yield tuple(members), open_member
        return
    with ZipFile(source) as archive:
        names = tuple(i.filename for i in archive.infolist() if not i.is_dir())
        if len(set(names)) != len(names):
            raise ValueError(f"Entry ZIP recovery duplikat: {source}")
        yield names, archive.open

def read_recovery_json(open_member, name):
    with open_member(name) as stream:
        return json.load(stream)

def source_layout(names, open_member, key):
    states = [name for name in names if PurePosixPath(name).name == "run_summary.json"]
    if len(states) != 1:
        raise ValueError("Sumber recovery perlu satu run_summary.json pada direktori run.")
    state_name = states[0]
    root = state_name.removesuffix("run_summary.json")
    state = read_recovery_json(open_member, state_name)["models"][key]
    if state["training"] != "completed":
        raise ValueError(f"Training sumber {key} belum lengkap.")
    if root + "provenance/experiment_spec.json" in names:
        spec_name = root + "provenance/experiment_spec.json"
        history = root + f"provenance/model_specs/{key}.json"
        gt_name = root + "validation/hard3.coco.json"
        manifest_name = root + "provenance/dataset_manifest.json"
        prefix = root + f"model/{key}/"
    else:
        spec_name = root + "experiment_spec.json"
        history = root + f"model_specs/{key}.json"
        gt_name = root + "dataset/valid/hard3.coco.json"
        manifest_name = root + "dataset_manifest.json"
        attempt = PurePosixPath(state["run"]).name
        if not re.fullmatch(r"attempt_\d+", attempt):
            raise ValueError(f"Nama percobaan tidak valid: {attempt}")
        prefix = root + f"experiments/{key}/{attempt}/"
    spec_name = history if history in names else spec_name
    for required in (spec_name, gt_name, manifest_name, prefix + "summary.json", prefix + "config.json"):
        if required not in names:
            raise ValueError(f"Entry sumber recovery tidak tersedia: {required}")
    return prefix, spec_name, gt_name, manifest_name, state

def is_recovery_directory(path):
    return ((path / "experiment_spec.json").is_file() and (path / "experiments").is_dir()
            or (path / "provenance/experiment_spec.json").is_file() and (path / "model").is_dir())

def discover_recovery_source():
    if RECOVERY_SOURCE_PATH is not None:
        explicit = Path(RECOVERY_SOURCE_PATH)
        if explicit.is_file() or (explicit.is_dir() and (explicit / "run_summary.json").is_file()):
            return explicit
        if not explicit.is_dir():
            raise FileNotFoundError(explicit)
        search_root = explicit
    else:
        search_root = INPUT_ROOT
    if not search_root.is_dir():
        raise FileNotFoundError(search_root)
    directories = sorted({p.parent for p in search_root.rglob("run_summary.json")
                          if is_recovery_directory(p.parent)})
    archives = sorted(p for p in search_root.rglob("*.zip")
                      if p.name.casefold() == "cpmk3_hard3_all_models.zip")
    if len(archives) == 1:
        return archives[0]
    if not archives and len(directories) == 1:
        return directories[0]
    if not archives:
        original = []
        for directory in directories:
            for name in ("experiment_spec.json", "provenance/experiment_spec.json"):
                spec = directory / name
                if spec.is_file():
                    maskrcnn = read_json(spec)["maskrcnn"]
                    if maskrcnn.get("amp") is True and maskrcnn.get("lr") == 0.005:
                        original.append(directory)
                    break
        if len(original) == 1:
            return original[0]
    raise RuntimeError(f"Perlu satu sumber Hard-3 pertama. Isi RECOVERY_SOURCE_PATH. "
                       f"Kandidat ZIP: {archives}. Kandidat direktori: {directories}.")

def verify_source_spec(key, source_spec):
    if key not in YOLO_WEIGHTS or set(source_spec) != set(EXPERIMENT_SPEC):
        raise ValueError("Skema spesifikasi recovery tidak sesuai.")
    for field in ("yolo", "classes", "source_hashes", "protocol"):
        if fingerprint(source_spec[field]) != fingerprint(EXPERIMENT_SPEC[field]):
            raise ValueError(f"Spesifikasi sumber YOLO berbeda pada {field}.")
    previous_rcnn = source_spec["maskrcnn"]
    allowed = {**RCNN_TRAIN, "lr": 0.005}
    del allowed["gradient_clip_max_norm"]
    if (type(previous_rcnn.get("amp")) is not bool
            or dict(previous_rcnn, amp=False) != allowed):
        raise ValueError("Spesifikasi sumber tidak cocok dengan run Hard-3 awal atau recovery FP32 pertama.")
    return fingerprint(source_spec)

def restore_model(key, source, run):
    with recovery_reader(source) as (names, open_member):
        prefix, spec_name, gt_name, manifest_name, state = source_layout(names, open_member, key)
        with open_member(spec_name) as stream:
            spec_bytes = stream.read()
        source_spec = json.loads(spec_bytes)
        spec_hash = verify_source_spec(key, source_spec)
        summary = read_recovery_json(open_member, prefix + "summary.json")
        if summary["spec_sha256"] != spec_hash:
            raise ValueError(f"Hash summary {key} tidak merujuk pada spesifikasi historis.")
        config = read_recovery_json(open_member, prefix + "config.json")
        weights, weight_hash = YOLO_WEIGHTS[key]
        if (any(config.get(k) != v for k, v in YOLO_TRAIN.items()) or config.get("classes") != NAMES
                or config.get("pretrained") != weights or config.get("pretrained_sha256") != weight_hash):
            raise ValueError(f"Konfigurasi model {key} tidak sesuai.")
        source_gt = read_recovery_json(open_member, gt_name)
        if fingerprint(source_gt) != fingerprint(read_json(VAL_ANN)):
            raise ValueError("Ground truth validation berbeda dari run YOLO yang dipulihkan.")
        historical_manifest = read_recovery_json(open_member, manifest_name)
        # Lokasi mount bisa berbeda. Pemetaan, split, dan koreksi harus tetap cocok.
        manifest_fields = set(MANIFEST) - {"source"}
        if any(fingerprint(historical_manifest[name]) != fingerprint(MANIFEST[name])
               for name in manifest_fields):
            raise ValueError("Provenance dataset historis berbeda.")
        for name in sorted(n for n in names if n.startswith(prefix)):
            relative = PurePosixPath(name.removeprefix(prefix))
            if (relative.is_absolute() or ".." in relative.parts or not relative.parts
                    or any("\\" in part or ":" in part for part in relative.parts)):
                raise ValueError(f"Path entry recovery tidak valid: {name}")
            path = run.joinpath(*relative.parts)
            path.parent.mkdir(parents=True, exist_ok=True)
            with open_member(name) as incoming, path.open("xb") as outgoing:
                shutil.copyfileobj(incoming, outgoing)
    validate_training_artifacts(key, run, spec_hash)
    spec_dir = ROOT / "model_specs"
    spec_dir.mkdir(exist_ok=True)
    (spec_dir / f"{key}.json").write_bytes(spec_bytes)
    MODEL_SPEC_HASHES[key] = spec_hash
    validation_name = PurePosixPath(state.get("validation_dir", "")).name
    if state.get("evaluation") == "completed" and re.fullmatch(r"unified_validation_\d+", validation_name):
        return run / validation_name
    return None

def provenance_entries():
    entries = common_entries()
    spec_dir = ROOT / "model_specs"
    if spec_dir.is_dir():
        entries.extend(run_entries(spec_dir, "provenance/model_specs"))
    return entries

def run_training(key, trainer=None):
    state = MODEL_STATE[key]
    if state["training"] == "completed":
        return
    state.pop("error", None)
    state.pop("training_zip", None)
    state.pop("recovered_from", None)
    if key == "maskrcnn" and not RUN_TRAIN[key]:
        state.update(training="skipped", evaluation="skipped")
        save_status()
        return
    run = next_run(key)
    state.update(training="running", evaluation="pending", run=str(run))
    save_status()
    try:
        if key in YOLO_WEIGHTS:
            assert not RUN_TRAIN[key], "Notebook recovery tidak melatih YOLO."
            historical_eval = restore_model(key, RECOVERY_SOURCE, run)
            state.update(recovered_from=str(RECOVERY_SOURCE),
                         source_spec_sha256=MODEL_SPEC_HASHES[key])
            if historical_eval is not None:
                RECOVERED_EVALUATIONS[key] = historical_eval
        else:
            seed_all()
            trainer(key, run)
            validate_training_artifacts(key, run)
            destination = EXPORTS / f"{key}_{run.name}.zip"
            archive_verified(destination, common_entries() + run_entries(run, f"model/{key}"))
            state["training_zip"] = str(destination)
        ACTIVE_RUNS[key] = run
        state["training"] = "completed"
    except Exception:
        message = traceback.format_exc()
        run.mkdir(parents=True, exist_ok=True)
        (run / "training_error.txt").write_text(message, encoding="utf-8")
        state.update(training="failed", evaluation="skipped", error=str(run / "training_error.txt"))
        print(message)
    finally:
        release_gpu()
        save_status()

RECOVERY_SOURCE = discover_recovery_source()
print("Sumber recovery:", RECOVERY_SOURCE)
run_training("yolo11n")

In [ ]:
run_training("yolov8n")

## 8. Pemeriksaan awal dan pelatihan Mask R-CNN
Pada recovery pertama, batch tanpa instance positif menghasilkan loss RPN objectness yang tinggi dan dua loss RoI bernilai NaN. Torchvision dapat memakai proposal latar belakang untuk batch dengan target kosong. Jika tidak ada RoI terpilih, klasifikasi menerima target kosong dan loss regresi membagi dengan jumlah sampel nol. Jumlah proposal pada kegagalan sebelumnya tidak tercatat. Pembaruan SGD yang tidak stabil, gradien besar, dan aktivasi atau parameter nonfinite juga masih merupakan kemungkinan. Geometri target diperiksa pada persiapan data dan sebelum setiap forward. [Implementasi RoI heads Torchvision](https://github.com/pytorch/vision/blob/main/torchvision/models/detection/roi_heads.py)
Model diagnostik memeriksa batch ID 153 dan 205 serta satu batch positif. Kedua probe menjalankan forward dan backward tanpa langkah optimizer. RNG Python, NumPy, dan PyTorch dipulihkan, lalu model diagnostik dilepas sebelum pembuatan model pelatihan. Kegagalan probe menghentikan pelatihan Mask R-CNN.
Pada pelatihan utama, norma gradien diperiksa dan dibatasi sesudah backward sebelum SGD memperbarui parameter. Diagnostik mencatat 100 batch pertama dan setiap kegagalan. Pemeriksaan pada bobot awal membantu membedakan kegagalan yang langsung muncul dari kegagalan yang baru muncul setelah beberapa pembaruan.


In [ ]:
def coco_evaluate(gt, records, kind):
    with redirect_stdout(io.StringIO()):
        if records:
            detected = gt.loadRes(deepcopy(records))
        else:
            detected = COCO()
            detected.dataset = dict(info={}, images=gt.dataset["images"],
                                    categories=gt.dataset["categories"], annotations=[])
            detected.createIndex()
        evaluator = COCOeval(gt, detected, kind)
        evaluator.params.imgIds = sorted(gt.getImgIds())
        evaluator.params.catIds = [1, 2, 3]
        evaluator.params.maxDets = [1, 10, PROTOCOL["max_detections"]]
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    return evaluator

def as_rle(mask):
    coded = mask_utils.encode(np.asfortranarray(mask, dtype=np.uint8))
    coded["counts"] = coded["counts"].decode("ascii")
    return coded

def append_detections(boxes, labels, scores, masks, image_id, shape, threshold, bbox, segm):
    if not (len(boxes) == len(labels) == len(scores) == len(masks)):
        raise ValueError("Panjang keluaran prediksi berbeda.")
    if len(boxes) > PROTOCOL["max_detections"]:
        raise ValueError("Batas deteksi per gambar terlampaui.")
    for (x1, y1, x2, y2), cls, score, mask in zip(boxes, labels, scores, masks):
        x1, y1, x2, y2, score = map(float, (x1, y1, x2, y2, score))
        if not np.isfinite([x1, y1, x2, y2, score]).all():
            raise FloatingPointError("Prediksi tidak finite.")
        if score < threshold:
            continue
        if int(cls) not in (1, 2, 3) or mask.shape != shape:
            raise ValueError(f"ID kelas atau ukuran mask tidak sesuai: {cls}, {mask.shape}, {shape}")
        h, w = shape
        x1, x2 = np.clip([x1, x2], 0, w)
        y1, y2 = np.clip([y1, y2], 0, h)
        if x2 <= x1 or y2 <= y1:
            continue
        common = dict(image_id=image_id, category_id=int(cls), score=score)
        box = [float(x1), float(y1), float(x2-x1), float(y2-y1)]
        bbox.append(dict(common, bbox=box))
        binary = mask >= PROTOCOL["mask_threshold"]
        if binary.any():
            segm.append(dict(common, segmentation=as_rle(binary)))

def coco_metrics(gt, box_predictions, mask_predictions):
    box = coco_evaluate(gt, box_predictions, "bbox")
    mask = coco_evaluate(gt, mask_predictions, "segm")
    metrics = dict(box_map50_95=float(box.stats[0]), box_map50=float(box.stats[1]),
                   mask_map50_95=float(mask.stats[0]), mask_map50=float(mask.stats[1]),
                   mask_map75=float(mask.stats[2]), mask_ar100=float(mask.stats[8]))
    per_class = []
    for idx, cls in enumerate(mask.params.catIds):
        data = mask.eval["precision"][:, :, idx, 0, -1]
        values = data[data >= 0]
        per_class.append(dict(hard3_coco_id=cls, class_name=NAMES[cls-1],
                              mask_ap50_95=float(values.mean()) if values.size else None))
    return metrics, per_class

def make_maskrcnn(pretrained, unified=False):
    args = dict(min_size=RCNN_TRAIN["min_size"], max_size=RCNN_TRAIN["max_size"],
                box_detections_per_img=100)
    if unified:
        args.update(box_score_thresh=PROTOCOL["confidence_threshold"],
                    box_nms_thresh=PROTOCOL["maskrcnn_nms_iou"])
    weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT if pretrained else None
    model = maskrcnn_resnet50_fpn(weights=weights, weights_backbone=None, **args)
    box_in = model.roi_heads.box_predictor.cls_score.in_features
    mask_in = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.box_predictor = FastRCNNPredictor(box_in, 4)
    model.roi_heads.mask_predictor = MaskRCNNPredictor(mask_in, 256, 4)
    assert model.roi_heads.box_predictor.cls_score.out_features == 4
    assert model.roi_heads.mask_predictor.mask_fcn_logits.out_channels == 4
    return model.to(DEVICE)

def validation_loader():
    return DataLoader(VAL_DS, batch_size=1, shuffle=False, num_workers=2,
                      pin_memory=True, collate_fn=collate_fn)

@torch.inference_mode()
def collect_maskrcnn(model, loader, threshold):
    box_predictions, mask_predictions, seen = [], [], set()
    model.eval()
    for images, targets in loader:
        outputs = model([im.to(DEVICE) for im in images])
        assert len(outputs) == len(images)
        for image, target, output in zip(images, targets, outputs):
            image_id = int(target["image_id"].item())
            assert image_id not in seen
            seen.add(image_id)
            masks = output["masks"].cpu().numpy()
            assert masks.shape == (len(output["labels"]), 1, *image.shape[-2:])
            append_detections(output["boxes"].cpu().numpy(), output["labels"].cpu().numpy(),
                              output["scores"].cpu().numpy(), masks[:, 0], image_id,
                              tuple(image.shape[-2:]), threshold, box_predictions, mask_predictions)
    assert seen == set(VAL_DS.ids)
    return box_predictions, mask_predictions

def save_rcnn_checkpoint(path, model, optimizer, scheduler, epoch, metrics):
    torch.save(dict(model_state_dict=model.state_dict(), optimizer_state_dict=optimizer.state_dict(),
                    scheduler_state_dict=scheduler.state_dict(), precision="float32",
                    epoch=epoch, metrics=metrics, classes=NAMES), path)

def batch_details(targets):
    return [{"image_id": int(t["image_id"].item()), "instances": len(t["labels"])}
            for t in targets]

def validate_rcnn_batch(images, targets):
    if not images or len(images) != len(targets):
        raise ValueError("Jumlah gambar dan target dalam batch tidak sesuai.")
    for image, target in zip(images, targets):
        image_id = int(target["image_id"].item())
        if image.ndim != 3 or image.shape[0] != 3 or image.dtype != torch.float32:
            raise ValueError(f"Input bukan RGB FP32: image_id={image_id}")
        if not torch.isfinite(image).all() or (image < 0).any() or (image > 1).any():
            raise ValueError(f"Nilai piksel di luar [0, 1]: image_id={image_id}")
        height, width = image.shape[-2:]
        labels, boxes, masks = target["labels"], target["boxes"], target["masks"]
        n = labels.numel()
        if (labels.shape != (n,) or labels.dtype != torch.int64
                or boxes.shape != (n, 4) or boxes.dtype != torch.float32
                or masks.shape != (n, height, width) or masks.dtype != torch.uint8
                or target["area"].shape != (n,) or target["iscrowd"].shape != (n,)):
            raise ValueError(f"Shape atau dtype target tidak sesuai: image_id={image_id}")
        if n == 0:
            continue  # Target kosong valid dan tetap diteruskan ke forward.
        if (not ((labels >= 1) & (labels <= 3)).all()
                or not torch.isfinite(boxes).all()
                or not ((boxes[:, 2:] - boxes[:, :2]) > 0).all()
                or (boxes[:, :2] < 0).any()
                or (boxes[:, 2] > width).any() or (boxes[:, 3] > height).any()
                or (masks > 1).any() or not masks.flatten(1).any(1).all()
                or not torch.isfinite(target["area"]).all() or (target["area"] <= 0).any()
                or (target["iscrowd"] != 0).any()):
            raise ValueError(f"Kategori, bbox, area, atau mask tidak valid: image_id={image_id}")

EXPECTED_RCNN_LOSSES = {
    "loss_classifier", "loss_box_reg", "loss_mask", "loss_objectness", "loss_rpn_box_reg",
}

def finite_or_text(number):
    return number if number is None or math.isfinite(number) else str(number)

def tensor_health(named_tensors):
    entries = [(name, tensor.detach()) for name, tensor in named_tensors if tensor.numel()]
    if not entries:
        return dict(checked_tensors=0, nonfinite=[], max_abs=None)
    peaks = torch.stack([t.abs().amax().float() for _, t in entries]).cpu().tolist()
    return dict(checked_tensors=len(entries),
                nonfinite=[name for (name, _), peak in zip(entries, peaks) if not math.isfinite(peak)],
                max_abs=max((peak for peak in peaks if math.isfinite(peak)), default=None))

def model_health(model):
    with torch.no_grad():
        return dict(parameters=tensor_health(model.named_parameters()),
                    floating_buffers=tensor_health(
                        (name, value) for name, value in model.named_buffers() if value.is_floating_point()
                    ))

@contextmanager
def proposal_trace(model):
    trace = {}

    def rpn_head_hook(_module, _inputs, output):
        logits, deltas = output
        trace["activations"] = {
            f"rpn_{name}_level_{level}": value.detach()
            for name, maps in (("objectness", logits), ("bbox_delta", deltas))
            for level, value in enumerate(maps)
        }

    def rpn_hook(_module, _inputs, output):
        boxes, _losses = output
        trace["rpn_proposals_per_image"] = [len(boxes_i) for boxes_i in boxes]

    def roi_pool_hook(_module, inputs):
        trace["roi_samples_per_image"] = [len(boxes_i) for boxes_i in inputs[1]]

    def roi_predictor_hook(_module, _inputs, output):
        scores, regressions = output
        trace.setdefault("activations", {}).update(
            roi_class_logits=scores.detach(), roi_bbox_delta=regressions.detach()
        )

    hooks = [
        model.rpn.head.register_forward_hook(rpn_head_hook),
        model.rpn.register_forward_hook(rpn_hook),
        model.roi_heads.box_roi_pool.register_forward_pre_hook(roi_pool_hook),
        model.roi_heads.box_predictor.register_forward_hook(roi_predictor_hook),
    ]
    try:
        yield trace
    finally:
        for hook in hooks:
            hook.remove()
        trace.clear()

def checked_rcnn_batch(model, images, targets, run, trace, epoch, batch, *,
                       optimizer=None, probe=None, log=None, step=None, inspect_state=False):
    details = batch_details(targets)
    record = dict(epoch=epoch, batch=batch, step=step, probe=probe, images=details,
                  batch_without_positive_instances=all(x["instances"] == 0 for x in details),
                  precision="float32", amp=False, loss_components=None, total_loss=None,
                  gradient_norm_before_clip=None, max_gradient_norm=RCNN_TRAIN["gradient_clip_max_norm"],
                  rpn_proposals_per_image=None, roi_samples_per_image=None)
    phase = "invalid_target"
    trace.clear()
    try:
        validate_rcnn_batch(images, targets)
        phase = "forward_failed"
        images = [image.to(DEVICE) for image in images]
        targets = [{k: v.to(DEVICE) for k, v in target.items()} for target in targets]
        model.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=False):
            parts = model(images, targets)
        if set(parts) != EXPECTED_RCNN_LOSSES or any(value.ndim != 0 for value in parts.values()):
            raise ValueError(f"Komponen loss Mask R-CNN berbeda: {list(parts)}")
        loss = sum(parts.values())
        values = {k: float(value.detach().item()) for k, value in parts.items()}
        total = float(loss.detach().item())
        record.update(loss_components={k: finite_or_text(v) for k, v in values.items()},
                      total_loss=finite_or_text(total))
        phase = "loss_nonfinite"
        if not math.isfinite(total) or not all(map(math.isfinite, values.values())):
            raise FloatingPointError("Komponen loss atau total loss nonfinite.")
        if log is not None:
            record["activations"] = tensor_health(trace.get("activations", {}).items())
            phase = "activation_nonfinite"
            if record["activations"]["nonfinite"]:
                raise FloatingPointError("Aktivasi RPN atau RoI nonfinite.")

        phase = "backward_failed"
        loss.backward()
        phase = "gradient_clipping_failed"
        parameters = [parameter for parameter in model.parameters() if parameter.grad is not None]
        if not parameters:
            raise RuntimeError("Backward tidak menghasilkan gradien.")
        # Norma L2 gabungan mendeteksi gradien NaN/Inf dan overflow saat reduksi.
        try:
            norm = torch.nn.utils.clip_grad_norm_(
                parameters, RCNN_TRAIN["gradient_clip_max_norm"], error_if_nonfinite=True,
            )
        except RuntimeError:
            norms = [torch.linalg.vector_norm(p.grad.detach()) for p in parameters]
            record["gradient_norm_before_clip"] = finite_or_text(
                float(torch.linalg.vector_norm(torch.stack(norms)).item())
            )
            raise
        record["gradient_norm_before_clip"] = finite_or_text(float(norm.item()))
        record["gradient_clipped"] = float(norm.item()) > RCNN_TRAIN["gradient_clip_max_norm"]
        if optimizer is not None:
            phase = "optimizer_step_failed"
            optimizer.step()
        if inspect_state:
            phase = "model_state_nonfinite"
            record["model_state"] = model_health(model)
            if any(health["nonfinite"] for health in record["model_state"].values()):
                raise FloatingPointError("Parameter atau buffer model nonfinite.")
    except Exception as error:
        record.update(status="failed", reason=phase, error=f"{type(error).__name__}: {error}")
        record.update({k: v for k, v in trace.items() if k != "activations"})
        try:
            record["activations"] = tensor_health(trace.get("activations", {}).items())
            record["model_state"] = model_health(model)
            record["gradients"] = tensor_health(
                (name, p.grad) for name, p in model.named_parameters() if p.grad is not None
            )
        except RuntimeError as diagnostic_error:
            record["inspection_error"] = str(diagnostic_error)
        path = run / "batch_diagnostic.json"
        write_json(path, record)
        print(json.dumps(record, ensure_ascii=False), flush=True)
        raise
    else:
        if log is not None:
            record.update({k: v for k, v in trace.items() if k != "activations"})
            record["status"] = "passed"
            print(json.dumps(record, ensure_ascii=False), file=log, flush=True)
            if step is None or step == 1 or step % 10 == 0:
                print(json.dumps(record, ensure_ascii=False), flush=True)
        return values
    finally:
        trace.clear()

def run_rcnn_preflight(dataset, run):
    # Elakkan penggunaan iterator DataLoader pelatihan saat memilih batch probe.
    python_rng, numpy_rng = random.getstate(), np.random.get_state()
    try:
        with torch.random.fork_rng(devices=list(range(torch.cuda.device_count()))):
            probe_model = make_maskrcnn(pretrained=True).float().train()
            try:
                lookup = {image_id: index for index, image_id in enumerate(dataset.ids)}
                negative = [dataset[lookup[i]] for i in PREFLIGHT_NEGATIVE_IDS]
                positives = [i for i in dataset.ids if dataset.coco.imgToAnns[i]][:RCNN_TRAIN["batch"]]
                if len(positives) != RCNN_TRAIN["batch"]:
                    raise ValueError("Gambar train positif untuk preflight tidak cukup.")
                positive = [dataset[lookup[i]] for i in positives]
                if any(len(target["labels"]) for _, target in negative):
                    raise ValueError(f"Target gambar {PREFLIGHT_NEGATIVE_IDS} seharusnya kosong.")
                if any(not len(target["labels"]) for _, target in positive):
                    raise ValueError(f"Gambar probe positif tidak memiliki instance target: {positives}")
                with proposal_trace(probe_model) as trace, (
                    run / "preflight.jsonl"
                ).open("x", encoding="utf-8") as log:
                    for index, (name, samples) in enumerate(
                        (("historical_empty_targets", negative), ("positive_targets", positive)), 1
                    ):
                        images, targets = collate_fn(samples)
                        checked_rcnn_batch(probe_model, images, targets, run, trace, None, index,
                                           probe=name, log=log, inspect_state=True)
            finally:
                del probe_model
                release_gpu()
    finally:
        random.setstate(python_rng)
        np.random.set_state(numpy_rng)

def train_maskrcnn(key, run):
    weight_dir = run / "weights"
    weight_dir.mkdir(parents=True)
    cfg = RCNN_TRAIN
    write_json(run / "config.json", dict(cfg, pretrained="Torchvision COCO DEFAULT", classes=NAMES,
                                         foreground_classes=3, classes_with_background=4, precision="float32",
                                         train_validation_score_threshold=0.05))
    train_ds = IndoorCOCODataset("train")
    run_rcnn_preflight(train_ds, run)
    train_loader = DataLoader(train_ds, batch_size=cfg["batch"], shuffle=True,
                              num_workers=cfg["workers"], pin_memory=True, collate_fn=collate_fn,
                              generator=torch.Generator().manual_seed(SEED))
    val_loader = validation_loader()
    assert cfg["amp"] is False
    model = make_maskrcnn(pretrained=True).float()
    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                                lr=cfg["lr"], momentum=cfg["momentum"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, cfg["milestones"], gamma=cfg["gamma"])
    history, best = [], -math.inf
    start = time.perf_counter()
    step = 0
    with (run / "training.log").open("x", encoding="utf-8") as log, (
        run / "early_training_batches.jsonl"
    ).open("x", encoding="utf-8") as early_log, proposal_trace(model) as trace:
        for epoch in range(1, cfg["epochs"] + 1):
            if epoch > 1 and time.perf_counter() - start >= cfg["max_minutes"] * 60:
                raise TimeoutError(f"Anggaran waktu baseline tercapai sebelum epoch {epoch}.")
            lr = optimizer.param_groups[0]["lr"]
            model.train()
            sums = Counter()
            empty_batches = 0
            train_start = time.perf_counter()
            for batch, (images, targets) in enumerate(train_loader, 1):
                step += 1
                empty_batches += int(all(not len(target["labels"]) for target in targets))
                early = step <= RCNN_LOG_BATCHES
                values = checked_rcnn_batch(
                    model, images, targets, run, trace, epoch, batch, optimizer=optimizer, step=step,
                    log=early_log if early else None, inspect_state=early or batch == len(train_loader),
                )
                sums.update(values)

            torch.cuda.synchronize()
            train_seconds = time.perf_counter() - train_start
            val_start = time.perf_counter()
            boxes, masks = collect_maskrcnn(model, val_loader, threshold=0.05)
            metrics, per_class = coco_metrics(VAL_DS.coco, boxes, masks)
            validation_seconds = time.perf_counter() - val_start
            scheduler.step()
            row = dict(epoch=epoch, time=time.perf_counter()-start,
                       train_seconds=train_seconds, validation_seconds=validation_seconds,
                       learning_rate=lr, empty_target_batches=empty_batches, train_total_loss=sum(sums.values())/len(train_loader),
                       **{k: v/len(train_loader) for k, v in sums.items()}, **metrics)
            history.append(row)
            pd.DataFrame(history).to_csv(run / "results.csv", index=False)
            if metrics["mask_map50_95"] > best:
                best, best_epoch = metrics["mask_map50_95"], epoch
                save_rcnn_checkpoint(weight_dir / "best.pth", model, optimizer, scheduler, epoch, metrics)
                pd.DataFrame(per_class).to_csv(run / "per_class_training_validation.csv", index=False)
            save_rcnn_checkpoint(weight_dir / "last.pth", model, optimizer, scheduler, epoch, metrics)
            if epoch % cfg["save_period"] == 0:
                shutil.copy2(weight_dir / "last.pth", weight_dir / f"epoch{epoch:03d}.pth")
            print(row, file=log, flush=True)
            print(f"Mask R-CNN epoch {epoch}/{cfg['epochs']} | loss {row['train_total_loss']:.4f} | mask AP {metrics['mask_map50_95']:.4f}")
    h = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
    axes[0].plot(h.epoch, h.train_total_loss)
    axes[0].set(xlabel="Epoch", ylabel="Loss train")
    axes[1].plot(h.epoch, 100 * h.mask_map50_95)
    axes[1].set(xlabel="Epoch", ylabel="Mask mAP50–95 validation (%)")
    fig.savefig(run / "training_curves.png", dpi=150)
    plt.close(fig)
    finish_summary(key, run, dict(epochs_completed=len(history), training_seconds=history[-1]["time"],
                                  training_time_scope="Mask R-CNN epoch loop including validation",
                                  best_epoch=best_epoch, highest_validation_mask_map=best, precision="float32", amp=False,
                                  learning_rate_initial=cfg["lr"], gradient_clip_max_norm=cfg["gradient_clip_max_norm"],
                                  checkpoint_selection="validation mask mAP50–95"))

run_training("maskrcnn", train_maskrcnn)

## 9. Unified COCOeval pada validation

Ketiga checkpoint dinilai pada 166 gambar yang sama. Evaluator memakai confidence minimum 0,001, maksimum 100 deteksi per gambar, dan mask pada resolusi asli. YOLO memakai `imgsz=640`, NMS IoU 0,70, dan `retina_masks=True` untuk memulihkan ukuran mask asli. Mask R-CNN memakai resize 640–960, NMS IoU 0,50, dan mask threshold 0,50. Pada Ultralytics, threshold 0,50 dipakai untuk mask keluaran yang sudah biner. Durasi pipeline mencakup pembacaan gambar, prediksi, pemrosesan keluaran, dan encoding mask. Waktu pemuatan checkpoint, COCOeval, dan penulisan JSON metrik dicatat di luar ruang lingkup durasi pipeline.

In [ ]:
@torch.inference_mode()
def collect_yolo(model):
    metadata = {p["file_name"]: p for p in VAL_DS.coco.imgs.values()}
    inputs = [str(VAL_DS.image_dir / VAL_DS.coco.imgs[i]["file_name"]) for i in VAL_DS.ids]
    boxes, masks, seen = [], [], set()
    outputs = model.predict(source=inputs, stream=True, batch=1, device=0,
                            imgsz=PROTOCOL["yolo_image_size"], retina_masks=True,
                            conf=PROTOCOL["confidence_threshold"], iou=PROTOCOL["yolo_nms_iou"],
                            max_det=PROTOCOL["max_detections"], agnostic_nms=False,
                            rect=True, save=False, verbose=False)
    for result in outputs:
        meta = metadata[Path(result.path).name]
        image_id = meta["id"]
        assert image_id not in seen
        seen.add(image_id)
        shape = (meta["height"], meta["width"])
        assert result.orig_shape == shape, (meta["file_name"], result.orig_shape, shape)
        if result.boxes is None or not len(result.boxes):
            continue
        if result.masks is None or tuple(result.masks.data.shape[1:]) != shape:
            raise ValueError(f"Mask YOLO tidak pada resolusi asli: {meta['file_name']}")
        append_detections(result.boxes.xyxy.cpu().numpy(),
                          result.boxes.cls.cpu().numpy().astype(int)+1,
                          result.boxes.conf.cpu().numpy(), result.masks.data.cpu().numpy(),
                          image_id, shape, PROTOCOL["confidence_threshold"], boxes, masks)
    assert seen == set(VAL_DS.ids)
    return boxes, masks

def evaluate_checkpoint(key, run, out):
    checkpoint = run / checkpoint_names(key)[0]
    if key == "maskrcnn":
        model = make_maskrcnn(pretrained=False, unified=True)
        state = torch.load(checkpoint, map_location="cpu", weights_only=True)
        assert state["classes"] == NAMES and 1 <= state["epoch"] <= RCNN_TRAIN["epochs"]
        model.load_state_dict(state["model_state_dict"], strict=True)
        del state
        loader = validation_loader()
    else:
        model = YOLO(str(checkpoint), task="segment")
        assert model.task == "segment" and [model.names[i] for i in range(len(model.names))] == NAMES
        assert model.model.model[-1].nc == 3
    torch.cuda.synchronize()
    started = time.perf_counter()
    if key == "maskrcnn":
        boxes, masks = collect_maskrcnn(model, loader, PROTOCOL["confidence_threshold"])
    else:
        boxes, masks = collect_yolo(model)
    torch.cuda.synchronize()
    seconds = time.perf_counter() - started
    del model
    write_json(out / "bbox.json", boxes)
    write_json(out / "segm.json", masks)
    metrics, per_class = coco_metrics(VAL_DS.coco, boxes, masks)
    assert all(v is not None and 0 <= v <= 1 for v in metrics.values())
    assert all(row["mask_ap50_95"] is not None for row in per_class)
    pd.DataFrame(per_class).to_csv(out / "per_class_ap.csv", index=False)
    write_json(out / "metrics.json", dict(metrics, model=key, inference_pipeline_seconds=seconds,
                                          checkpoint_sha256=sha256(checkpoint), **PROTOCOL))
    for name in ("bbox.json", "segm.json", "per_class_ap.csv", "metrics.json"):
        assert (out / name).stat().st_size > 0

def validate_recovered_evaluation(key, run, out):
    metrics = read_json(out / "metrics.json")
    checkpoint_digest = read_json(run / "summary.json")["checkpoint_sha256"][checkpoint_names(key)[0]]
    if (metrics.get("model") != key or metrics.get("checkpoint_sha256") != checkpoint_digest
            or any(metrics.get(k) != v for k, v in PROTOCOL.items())):
        raise ValueError("Protokol evaluasi atau checkpoint historis berbeda.")
    seconds = float(metrics["inference_pipeline_seconds"])
    if not math.isfinite(seconds) or seconds <= 0:
        raise ValueError("Durasi evaluasi historis tidak valid.")
    for name in ("mask_map50_95", "mask_map50", "mask_map75", "mask_ar100", "box_map50_95", "box_map50"):
        if not 0 <= float(metrics[name]) <= 1:
            raise ValueError(f"Metrik historis tidak valid: {name}")
    with (out / "per_class_ap.csv").open(encoding="utf-8-sig", newline="") as stream:
        table = list(csv.DictReader(stream))
    if (len(table) != 3
            or {int(r["hard3_coco_id"]): r["class_name"] for r in table} != dict(enumerate(NAMES, 1))):
        raise ValueError("Pemetaan kelas historis berbeda.")
    scores = [float(r["mask_ap50_95"]) for r in table]
    if (not all(0 <= v <= 1 for v in scores)
            or not math.isclose(sum(scores)/len(scores), metrics["mask_map50_95"], abs_tol=1e-10)):
        raise ValueError("Rata-rata AP per kelas berbeda dari mask mAP50–95 historis.")
    metadata = VAL_DS.coco.imgs
    for kind in ("bbox", "segm"):
        records = read_json(out / f"{kind}.json")
        seen = Counter()
        for row in records:
            image = metadata[row["image_id"]]
            if (row["category_id"] not in (1, 2, 3)
                    or not PROTOCOL["confidence_threshold"] <= row["score"] <= 1):
                raise ValueError("Kelas atau confidence prediksi historis tidak valid.")
            seen[row["image_id"]] += 1
            if kind == "bbox":
                x, y, w, h = row["bbox"]
                if not (min(x, y) >= 0 and min(w, h) > 0
                        and x + w <= image["width"] + 1e-6 and y + h <= image["height"] + 1e-6):
                    raise ValueError("BBox historis di luar gambar.")
            elif row["segmentation"]["size"] != [image["height"], image["width"]]:
                raise ValueError("Ukuran mask historis berbeda dari resolusi asli.")
        if max(seen.values(), default=0) > PROTOCOL["max_detections"]:
            raise ValueError("Batas deteksi historis terlampaui.")

def run_evaluation(key):
    state = MODEL_STATE[key]
    if state["evaluation"] == "completed":
        return
    if not RUN_EVALUATION or state["training"] != "completed":
        state["evaluation"] = "skipped"
        save_status()
        return
    run = ACTIVE_RUNS[key]
    if key in RECOVERED_EVALUATIONS:
        historical = RECOVERED_EVALUATIONS[key]
        try:
            validate_recovered_evaluation(key, run, historical)
        except (OSError, ValueError, KeyError, TypeError) as error:
            print(f"{key}: evaluasi historis tidak lolos pemeriksaan ({error}). Evaluasi ulang.")
        else:
            state.update(evaluation="completed", validation_dir=str(historical), evaluation_origin="recovered")
            save_status()
            return
    idx = 1
    while (run / f"unified_validation_{idx:02d}").exists():
        idx += 1
    out = run / f"unified_validation_{idx:02d}"
    out.mkdir()
    state.update(evaluation="running", validation_dir=str(out))
    save_status()
    try:
        evaluate_checkpoint(key, run, out)
        state.update(evaluation="completed", evaluation_origin="current_run")
    except Exception:
        message = traceback.format_exc()
        (out / "evaluation_error.txt").write_text(message, encoding="utf-8")
        state.update(evaluation="failed", evaluation_error=str(out / "evaluation_error.txt"))
        print(message)
    finally:
        release_gpu()
        save_status()

for key in RUN_TRAIN:
    run_evaluation(key)

## 10. Perbandingan all-class dan Hard-3

Tabel menggunakan tiga kelas yang sama. Kecocokan sumber baseline, parameter evaluasi yang tercatat, dan geometri ground truth tiga kelas diperiksa sebelum selisih dihitung. Baseline menyediakan AP dari model all-class, dengan pengaturan jumlah deteksi yang diterapkan saat model masih memprediksi 74 kelas. Susunan kelas, bobot head, dan persaingan deteksi merupakan bagian dari perbedaan eksperimen. Notebook evaluator all-class belum tersedia di repo, sehingga kesetaraan implementasi detail rasterisasi dan pascapemrosesan belum dapat dipastikan. Selisih pada validation bersifat deskriptif.

In [ ]:
BASELINE_FINGERPRINTS = {"config":"8d61941fbdfeb22a9b74e7b48dc154d73108c9025b86f4511e35adc9e5482feb","per_class":"6be50420eaf387c73534f101ceac59e3eac89ca486ac4b771d8f2e96e15d08c9"}
EXPECTED_VAL_SIGNATURE = "1328aacbf05d7e2666500494f7a17493f7859eaea326b12671fde23ba441d72b"

def validation_signature(gt, convert_ids):
    images = sorted((a["id"], a["file_name"], a["width"], a["height"])
                    for a in gt["images"])
    annotations = sorted(
        (a["id"], a["image_id"], convert_ids[a["category_id"]],
         [[round(float(x), 8) for x in polygon] for polygon in a["segmentation"]])
        for a in gt["annotations"] if a["category_id"] in convert_ids
    )
    return fingerprint([images, annotations])

def evaluate_baseline_payload(payload):
    cfg = json.loads(payload["evaluation_config.json"])
    rows = list(csv.DictReader(io.StringIO(payload["per_class_ap_comparison.csv"].decode("utf-8-sig"))))
    if dict(config=fingerprint(cfg), per_class=fingerprint(rows)) != BASELINE_FINGERPRINTS:
        raise ValueError("Artefak baseline berbeda dari hasil unified COCOeval yang diaudit.")
    assert all(cfg[k] == v for k, v in PROTOCOL.items())
    assert cfg["num_classes"] == 74 and cfg["retraining_performed"] is False
    assert cfg["test_set_evaluated"] is False
    signatures = {(r["image_id"], r["file_name"], r["rotation"]) for r in cfg["orientation_fixes"]}
    assert signatures == {(r["image_id"], r["file_name"], r["rotation"])
                          for r in MANIFEST["orientation_repairs"] if r["split"] == "valid"}
    assert validation_signature(read_json(VAL_ANN), {n: n for n in range(1, 4)}) == EXPECTED_VAL_SIGNATURE
    selected = {int(r["category_id"]): r for r in rows if int(r["category_id"]) in CLASSES}
    assert set(selected) == set(CLASSES)
    for old, (new, name) in CLASSES.items():
        r = selected[old]
        assert r["class_name"] == name
        assert int(r["train_instances"]) == MANIFEST["splits"]["train"]["instances_by_class"][new]
        assert int(r["val_instances"]) == MANIFEST["splits"]["valid"]["instances_by_class"][new]
        assert all(0 <= float(r[key+"_mask_ap"]) <= 1 for key in RUN_TRAIN)
    return selected

def baseline_payload(source):
    names = ("evaluation_config.json", "per_class_ap_comparison.csv")
    if source.is_dir():
        return {k: (source / k).read_bytes() for k in names}
    with ZipFile(source) as z:
        data = {}
        for basename in names:
            matches = [n for n in z.namelist() if PurePosixPath(n).name == basename]
            if len(matches) != 1:
                raise ValueError(f"Perlu satu {basename} dalam {source}")
            data[basename] = z.read(matches[0])
    return data

def discover_baseline():
    if BASELINE_SOURCE is not None:
        return Path(BASELINE_SOURCE)
    candidates = {p.parent for p in INPUT_ROOT.rglob("per_class_ap_comparison.csv")
                  if (p.parent / "evaluation_config.json").is_file()}
    candidates.update(p for p in INPUT_ROOT.rglob("*.zip")
                      if p.name.lower() in {"hard-3 selection.zip", "all_class_evaluation_outputs.zip"})
    verified, failures = [], []
    for candidate in sorted(candidates):
        try:
            evaluate_baseline_payload(baseline_payload(candidate))
        except (OSError, ValueError, KeyError, AssertionError, BadZipFile) as error:
            failures.append(f"{candidate}: {error}")
        else:
            verified.append(candidate)
    if not verified:
        raise RuntimeError(f"Sumber baseline yang cocok tidak ditemukan: {failures}")
    return verified[0]  # Semua kandidat yang lolos memiliki fingerprint isi yang sama.

COMPARISON = ROOT / "comparison"
COMPARISON.mkdir()
comparison_status = "unavailable"
try:
    source = discover_baseline()
    payload = baseline_payload(source)
    baseline = evaluate_baseline_payload(payload)
    for name, contents in payload.items():
        (COMPARISON / ("all_class_" + name)).write_bytes(contents)
    by_class, by_model = [], []
    for key in RUN_TRAIN:
        state = MODEL_STATE[key]
        if state["evaluation"] != "completed":
            continue
        out = Path(state["validation_dir"])
        ev = read_json(out / "metrics.json")
        train = read_json(ACTIVE_RUNS[key] / "summary.json")
        hard = pd.read_csv(out / "per_class_ap.csv").set_index("hard3_coco_id")
        model_rows = []
        for old, (new, name) in CLASSES.items():
            ref = float(baseline[old][key + "_mask_ap"])
            score = float(hard.loc[new, "mask_ap50_95"])
            assert hard.loc[new, "class_name"] == name
            model_rows.append(dict(model=key, class_name=name, original_coco_id=old,
                                   hard3_coco_id=new, all_class_ap_pct=100*ref,
                                   hard3_ap_pct=100*score, delta_pp=100*(score-ref)))
        by_class.extend(model_rows)
        ref_epochs, ref_time = TRAIN_BASELINE[key]
        by_model.append(dict(
            model=key, all_class_mean_ap_pct=np.mean([r["all_class_ap_pct"] for r in model_rows]),
            hard3_mean_ap_pct=np.mean([r["hard3_ap_pct"] for r in model_rows]),
            delta_pp=np.mean([r["delta_pp"] for r in model_rows]),
            all_class_epochs=ref_epochs, hard3_epochs=train["epochs_completed"],
            all_class_training_seconds=ref_time, hard3_training_seconds=train["training_seconds"],
            all_class_inference_pipeline_seconds=PIPELINE_BASELINE[key],
            hard3_inference_pipeline_seconds=ev["inference_pipeline_seconds"],
        ))
    if by_model:
        details = pd.DataFrame(by_class)
        means = pd.DataFrame(by_model)
        details.to_csv(COMPARISON / "per_class_comparison.csv", index=False)
        means.to_csv(COMPARISON / "model_comparison.csv", index=False)
        display(means.round(3))
        display(details.round(3))
        comparison_status = "complete" if len(by_model) == len(RUN_TRAIN) else "partial"
    write_json(COMPARISON / "protocol_check.json", dict(
        baseline_source=str(source), config_matches=True, foreground_ground_truth_matches=True,
        includes_test_metrics=False,
        maskrcnn_training_difference=dict(baseline_amp=True, baseline_lr=0.005,
                                          hard3_amp=False, hard3_lr=RCNN_TRAIN["lr"],
                                          hard3_gradient_clip_max_norm=RCNN_TRAIN["gradient_clip_max_norm"]),
        limitation="Kesetaraan kode evaluator all-class tidak dapat diaudit tanpa notebook asli.",
        baseline_training_time_scope="Waktu kumulatif dari CSV baseline",
        baseline_training_source="experiments/<model>_all_classes/metrics/results.csv",
    ))
except Exception:
    error = traceback.format_exc()
    (COMPARISON / "comparison_error.txt").write_text(error, encoding="utf-8")
    print(error)

## 11. Grafik dan ekspor
Grafik membandingkan Mask AP50–95 pada tiga kategori yang sama. Perbedaan precision, laju belajar, dan gradient clipping pada Mask R-CNN membatasi atribusi selisih AP terhadap pengurangan kelas. Paket gabungan memuat hasil yang tersedia, status setiap model, dan diagnostik recovery. Jika preflight atau pelatihan Mask R-CNN gagal, hasil YOLO tetap diekspor dengan status keseluruhan `partial`.


In [ ]:
comparison_path = COMPARISON / "model_comparison.csv"
if comparison_path.is_file():
    matches = pd.read_csv(comparison_path).set_index("model")
    ax = matches[["all_class_mean_ap_pct", "hard3_mean_ap_pct"]].rename(columns={
        "all_class_mean_ap_pct": "All-class pada 3 kelas", "hard3_mean_ap_pct": "Hard-3",
    }).plot.bar(rot=0, color=["#356e9e", "#bd692f"], figsize=(8, 4))
    ax.set(ylabel="Mean Mask AP50–95 (%)", xlabel="Model", title="Perbandingan pada validation")
    ax.set_ylim(bottom=0)
    ax.figure.tight_layout()
    ax.figure.savefig(COMPARISON / "model_comparison.png", dpi=160)
    plt.show()
    plt.close(ax.figure)
    scores = pd.read_csv(COMPARISON / "per_class_comparison.csv")
    ax = scores.pivot(index="class_name", columns="model", values="delta_pp").plot.bar(rot=0)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(ylabel="Hard-3 − all-class (poin persentase)", xlabel="Kelas")
    ax.figure.tight_layout()
    ax.figure.savefig(COMPARISON / "per_class_delta.png", dpi=160)
    plt.show()
    plt.close(ax.figure)

all_complete = comparison_status == "complete" and all(
    s["training"] == "completed" and s["evaluation"] == "completed" for s in MODEL_STATE.values()
)
write_json(ROOT / "run_summary.json", dict(
    state="completed" if all_complete else "partial",
    models=MODEL_STATE, comparison=comparison_status, original_test_evaluated=False,
    personal_test_evaluated=False,
))
combined = provenance_entries() + [
    ("status.json", ROOT / "status.json"), ("run_summary.json", ROOT / "run_summary.json"),
    ("validation/hard3.coco.json", VAL_ANN),
] + run_entries(COMPARISON, "comparison")
for key, run in ACTIVE_RUNS.items():
    validate_training_artifacts(key, run, MODEL_SPEC_HASHES.get(key))
    combined.extend(run_entries(run, f"model/{key}"))
for key, state in MODEL_STATE.items():
    if state["training"] == "failed" and state.get("error"):
        error_file = Path(state["error"])
        combined.append((f"errors/{key}.txt", error_file))
        for name in ("batch_diagnostic.json", "preflight.jsonl", "early_training_batches.jsonl",
                     "config.json", "training.log", "results.csv"):
            diagnostic = error_file.with_name(name)
            if diagnostic.is_file():
                combined.append((f"errors/{key}/{name}", diagnostic))
final_zip = EXPORTS / "CPMK3_Hard3_Recovery_V2.zip"
archive_verified(final_zip, combined)
print(f"Status keseluruhan: {'completed' if all_complete else 'partial'}")
print("ZIP gabungan:", final_zip)
display(pd.DataFrame.from_dict(MODEL_STATE, orient="index"))
if not all_complete:
    print("RUN PARSIAL: periksa status.json dan traceback. Simpan versi ini untuk memulihkan run yang lengkap.")